# Phase 13 — Tuning the Pocket-Center Localization Step

Phase 12 found the model loses to P2Rank on DCC/DCA (pocket-center accuracy)
despite beating it on per-residue PR-AUC (Phase 11). The suspected cause: the
per-residue-probabilities-to-single-point conversion was an untuned heuristic
(fixed thresholds, connected components, unweighted centroid) bolted onto a
model that was never trained for localization — unlike P2Rank, which is
purpose-built for exactly this output.

This notebook does not touch the trained model at all. It reuses the exact
Phase 12 ensemble checkpoints and asks a narrower question: **given the same
per-residue predictions, can a better-designed conversion step close some of
the gap?**

### Discipline: tune on one half, report on the other, never both

Re-tuning a heuristic and reporting its score on the same data it was tuned on
is the exact mistake this project has spent many phases avoiding elsewhere. The
1,149 leakage-clean benchmark structures from Phase 12 are split by a fixed
seed into a **35% tune set** (used only to pick the best heuristic) and a
**65% test set** (scored exactly once, with the heuristic already fixed). The
untuned Phase-12 heuristic is also re-scored on this same test slice, so the
before/after comparison is paired and fair — not "old number on 1,149
structures vs new number on a different 650."

### Why several candidate heuristics, not just re-tuned thresholds

The working hypothesis is that connected-components clustering at a loose
threshold can produce a large, diffuse blob whose centroid drifts away from the
true (small, tight) ligand pocket — a few residues at the "wrong end" of a big
component are enough to fail a strict 4 Å threshold. Two structurally different
families are tried, not just parameter variations on the original idea:

- **Top-K peak**: ignore spatial clustering entirely, take the K
  highest-probability residues, centroid them (weighted or unweighted). Simple,
  and immune to the diffuse-blob failure mode by construction.
- **Connected components, revised**: the Phase 12 approach, but with a
  probability-*weighted* centroid instead of a plain mean — pulling the estimate
  toward the cluster's most confident core rather than treating a fringe residue
  the same as the peak.

## Cell 1 — Setup (identical to Phase 12)

In [ ]:
!pip install -q torch-geometric==2.6.1 gudhi==3.10.1 biopython==1.84 transformers==4.46.3
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y default-jre-headless > /dev/null 2>&1

In [ ]:
import csv as _csv
import gzip, json, os, shutil, subprocess, time, urllib.request, warnings
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore")

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob10 = torch.load(locate("phase10_graphs.pt"), map_location="cpu", weights_only=False)
blob6b = torch.load(locate("phase6b_graphs.pt"), map_location="cpu", weights_only=False)
train_dataset = blob10["dataset"]
N_PRE = int(blob10["n_pre_topo"]); TOPO_DIM = int(blob10["topo_dim"])
POS_WEIGHT = float(blob10["pos_weight"])
TOPO_NAMES = blob6b["topo_names"]

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")
print(f"training set: {len(train_dataset):,} chains, node dim {train_dataset[0].x.size(1)}")

WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

## Cell 2 — Fetch benchmark structures (identical to Phase 12)

In [ ]:
def parse_ds(url, prefix):
    lines = urllib.request.urlopen(url, timeout=60).read().decode().splitlines()
    out = []
    for ln in lines:
        ln = ln.strip()
        if not ln or ln.startswith("#"): continue
        name = ln.split("/")[-1].replace(".pdb", "")
        if prefix == "coach420":
            out.append((name[:4].upper(), name[4:].upper() or None))   # (pdb_id, chain or None)
        else:
            out.append((name[:4].upper(), None))
    return out

coach = parse_ds("https://raw.githubusercontent.com/rdk/p2rank-datasets/master/coach420.ds", "coach420")
holo  = parse_ds("https://raw.githubusercontent.com/rdk/p2rank-datasets/master/holo4k.ds", "holo4k")
print(f"COACH420: {len(coach)} entries | HOLO4K: {len(holo)} entries")

bench = [("coach420", pid, ch) for pid, ch in coach] + [("holo4k", pid, ch) for pid, ch in holo]
need = sorted({pid for _, pid, _ in bench})
print(f"unique PDB IDs to fetch: {len(need):,}")

def fetch(pid):
    p = RAW / f"{pid}.pdb.gz"
    if p.exists() and p.stat().st_size > 500: return pid, p
    try:
        with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.pdb.gz", timeout=60) as r:
            b = r.read()
        if len(b) > 500: p.write_bytes(b); return pid, p
    except Exception: pass
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=24) as ex: got = list(ex.map(fetch, need))
paths = {p: f for p, f in got if f}
print(f"fetched {len(paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

## Cell 3 — Ground-truth ligand extraction + leakage check (identical to Phase 12)

In [ ]:
from Bio.PDB import PDBParser

AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}
NON_LIGAND={"HOH","DOD","WAT","SO4","PO4","GOL","EDO","PEG","PG4","PGE","1PE","MPD","TRS",
            "ACT","ACY","FMT","CIT","MES","EPE","DMS","IMD","BME","NO3","AZI","CO3","IOD",
            "BR","CL","F","NA","K","CS","RB","LI","UNX","UNL","UNK"}
CATALYTIC_METALS={"ZN","MG","MN","FE","FE2","CU","CU1","CA","NI","CO","CD","W","MO"}
MIN_RES, MAX_RES, MIN_LIG_ATOMS = 50, 1000, 6

train_pdb_ids = {d.pdb_id for d in train_dataset}
overlap_exact = {pid for _, pid, _ in bench if pid in train_pdb_ids}
print(f"exact PDB-ID overlap with training set: {len(overlap_exact)}/{len(need)} unique IDs")

def parse_structure(pid, path, wanted_chain):
    """Returns list of (chain_id, seq, ca_xyz, per_res_atoms, ligand_list)."""
    try:
        with gzip.open(path, "rt") as fh:
            model = next(PDBParser(QUIET=True, PERMISSIVE=True).get_structure(pid, fh).get_models())
    except Exception:
        return []
    ligs = []
    for ch in model:
        for res in ch:
            if res.id[0] == " ": continue
            nm = res.get_resname().strip().upper()
            if nm in AA3_TO_1 or nm in NON_LIGAND: continue
            xyz = np.asarray([a.coord for a in res.get_unpacked_list() if a.element not in ("H","D")],
                            dtype=np.float32).reshape(-1,3)
            is_metal = nm in CATALYTIC_METALS
            if is_metal or len(xyz) >= MIN_LIG_ATOMS:
                ligs.append(xyz)
    out = []
    chains = [c for c in model] if wanted_chain is None else \
             [model[wanted_chain]] if wanted_chain in [c.id for c in model] else []
    for ch in chains:
        residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
        if not (MIN_RES <= len(residues) <= MAX_RES): continue
        ca=[r["CA"].coord for r in residues]
        seq="".join(AA3_TO_1.get(r.get_resname().strip().upper(),"X") for r in residues)
        atoms=[np.asarray([a.coord for a in r.get_unpacked_list() if a.element not in ("H","D")],
                          dtype=np.float32).reshape(-1,3) for r in residues]
        out.append((ch.id, seq, np.asarray(ca,dtype=np.float32), atoms, ligs))
    return out

t0=time.time()
parsed = {}   # pdb_id -> list of chain tuples, ALL chains parsed once per structure
for pid in sorted(paths):
    parsed[pid] = parse_structure(pid, paths[pid], None)
print(f"parsed {len(parsed)} structures in {(time.time()-t0)/60:.1f} min")

# assemble the evaluation unit list: one entry per (benchmark, pdb_id, target_chain_or_None)
eval_units = []
for src, pid, ch in bench:
    chains = parsed.get(pid, [])
    if not chains: continue
    if src == "coach420":
        match = [c for c in chains if c[0] == ch]
        if match: eval_units.append((src, pid, [match[0]]))
    else:
        # HOLO4K evaluates the whole assembly, but some entries are large
        # multi-chain complexes (viral capsids etc.) -- cap to the 6 longest
        # chains so one pathological structure cannot dominate runtime. This
        # covers the large majority of biologically relevant subunits; the
        # count actually capped is reported below rather than silently dropped.
        if chains:
            capped = sorted(chains, key=lambda c: -len(c[1]))[:6]
            eval_units.append((src, pid, capped))
n_capped = sum(1 for src,pid,ch in eval_units if src=="holo4k"
              and len(parsed.get(pid,[])) > len(ch))
print(f"evaluable units (structure parsed, target chain present): {len(eval_units)}/{len(bench)}")
print(f"HOLO4K structures with >6 chains, capped to 6 longest: {n_capped}")

In [ ]:
# --- MMseqs2 homology check against the training set --------------------------
mm = None
try:
    subprocess.run("curl -sL https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz -o /tmp/mm.tar.gz "
                   "&& tar xzf /tmp/mm.tar.gz -C /tmp", shell=True, check=True, timeout=600)
    c = Path("/tmp/mmseqs/bin/mmseqs")
    if c.exists(): os.chmod(c, 0o755); mm = str(c)
except Exception as e:
    print("mmseqs2 unavailable:", type(e).__name__)

W = Path("/tmp/leak_mm"); shutil.rmtree(W, ignore_errors=True); W.mkdir()
fasta_lines = []
for d in train_dataset:
    fasta_lines.append(f">TRAIN__{d.pdb_id}_{d.chain_id}\n{d.seq}\n")
bench_seq_map = {}
for src, pid, chains in eval_units:
    for cid, seq, *_ in chains:
        key = f"BENCH__{src}__{pid}_{cid}"
        bench_seq_map[key] = (src, pid, cid)
        fasta_lines.append(f">{key}\n{seq}\n")
(W/"all.fasta").write_text("".join(fasta_lines))

subprocess.run([mm, "easy-cluster", str(W/"all.fasta"), str(W/"res"), str(W/"tmp"),
                "--min-seq-id", "0.30", "-c", "0.8", "--cov-mode", "1", "-v", "1"],
               check=True, capture_output=True, timeout=3600)
rep = defaultdict(list)
for line in (W/"res_cluster.tsv").read_text().splitlines():
    a, b = line.split("\t"); rep[a].append(b)
clusters = list(rep.values()) if rep else []
# merge singleton reps (mmseqs always includes the rep itself in its own cluster list)
for r_, members in rep.items():
    if r_ not in members: members.append(r_)

leaked_keys = set()
for members in rep.values():
    has_train = any(m.startswith("TRAIN__") for m in members)
    has_bench = any(m.startswith("BENCH__") for m in members)
    if has_train and has_bench:
        leaked_keys.update(m for m in members if m.startswith("BENCH__"))

print(f"benchmark chains flagged as homologous (>=30% identity) to a training chain: "
      f"{len(leaked_keys)}/{len(bench_seq_map)}")

clean_units = []
for src, pid, chains in eval_units:
    clean_chains = [c for c in chains if f"BENCH__{src}__{pid}_{c[0]}" not in leaked_keys
                    and pid not in overlap_exact]
    if clean_chains:
        clean_units.append((src, pid, clean_chains))
n_dropped = len(eval_units) - len(clean_units)
print(f"\nclean (non-leaked) evaluation units: {len(clean_units)}/{len(eval_units)} "
      f"({n_dropped} dropped to leakage)")
for name in ("coach420","holo4k"):
    tot = sum(1 for s,_,_ in eval_units if s==name)
    cln = sum(1 for s,_,_ in clean_units if s==name)
    print(f"  {name:<10} {cln}/{tot} clean")

## Cell 4 — Load the Phase 12 ensemble (skip retraining)

Same three checkpoints Phase 12 trained on all 2,056 chains. Loading them here
instead of retraining saves roughly 9 minutes and, more importantly, guarantees
this is testing a heuristic change against a **fixed** model rather than
introducing fresh training-seed variance into the comparison.

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x
class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)
class Net(nn.Module):
    def __init__(s,n_pre,td,h=128,L=4,dr=0.1):
        super().__init__(); s.n_pre=n_pre
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
    def forward(s,d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        return s.head(h).squeeze(-1)

def run_epoch(model, loader, opt=None):
    tr = opt is not None; model.train(tr)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    ps,ys=[],[]
    for b in loader:
        b=b.to(DEVICE)
        with torch.set_grad_enabled(tr):
            lg=model(b); loss=crit(lg,b.y)
        if tr:
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        ps.append(torch.sigmoid(lg).detach().cpu().numpy()); ys.append(b.y.cpu().numpy())
    from sklearn.metrics import average_precision_score
    p,y=np.concatenate(ps),np.concatenate(ys)
    return {"pr": average_precision_score(y,p) if y.sum() else float("nan")}

def locate_any(fname):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(fname))
    return c

ensemble = []
for seed in (0,1,2):
    cands = locate_any(f"deploy_seed{seed}.pt")
    assert cands, f"deploy_seed{seed}.pt not found -- attach the Phase 12 kernel output"
    state = torch.load(max(cands, key=lambda p: p.stat().st_size), map_location=DEVICE)
    m = Net(N_PRE, TOPO_DIM).to(DEVICE); m.load_state_dict(state); m.eval()
    ensemble.append(m)
print(f"loaded {len(ensemble)} ensemble members from Phase 12 (no retraining)")

## Cell 5 — Featurize the clean benchmark chains (identical to Phase 12)

In [ ]:
import gudhi
from transformers import AutoTokenizer, AutoModel
from Bio.PDB import PPBuilder

AA_ALPHABET="ACDEFGHIKLMNPQRSTVWYX"; A2I={a:i for i,a in enumerate(AA_ALPHABET)}
MAX_HOM_DIM=2; BETTI_RADII=(2.0,3.0,5.0); NBHD_RADIUS=12.0; MAX_NBHD_ATOMS=300
FPD=6+2+len(BETTI_RADII)

# Phase 6b pruned one constant column (35 raw features -> 34) before the model
# ever saw them. Reconstruct the full 35-name list with the SAME generation
# logic used in every prior phase, then match by NAME against blob6b's saved
# 34-name list to recover which column was dropped -- matching by name (not a
# hardcoded index) means this self-verifies via the assert rather than silently
# drifting if the pruned column ever changes.
_FULL_TOPO_NAMES = [f"H{d}_{s}" for d in range(MAX_HOM_DIM+1)
                    for s in ["n_bars","total_pers","max_pers","mean_pers","std_pers","entropy",
                              "mean_birth","mean_death", *[f"betti_r{r:g}" for r in BETTI_RADII]]]                    + ["nbhd_n_atoms","nbhd_gyration"]
_TOPO_KEEP = np.array([n in set(TOPO_NAMES) for n in _FULL_TOPO_NAMES])
assert _TOPO_KEEP.sum() == TOPO_DIM,     f"expected {TOPO_DIM} surviving topo columns, matched {_TOPO_KEEP.sum()} by name"
print(f"topo pruning reconstructed: {len(_FULL_TOPO_NAMES)} raw -> {_TOPO_KEEP.sum()} kept "
      f"(dropped: {[n for n,k in zip(_FULL_TOPO_NAMES,_TOPO_KEEP) if not k]})")

def rgc(pos,r=8.0,k=32):
    n=pos.size(0); pos=pos-pos.mean(0,keepdim=True)
    d=torch.cdist(pos,pos).clone(); d.fill_diagonal_(float("inf")); m=d<=r
    if k<n:
        _,i=torch.topk(d,k,dim=1,largest=False); keep=torch.zeros_like(m); keep.scatter_(1,i,True); m&=keep
    m=m|m.t(); t,s=m.nonzero(as_tuple=True); return torch.stack([s,t]).long()

def _dg(pts):
    pts=np.ascontiguousarray(pts,dtype=np.float64); st=None; sq=True
    if len(pts)>=5:
        try: st=gudhi.AlphaComplex(points=pts).create_simplex_tree()
        except Exception: st=None
    if st is None:
        st=gudhi.RipsComplex(points=pts,max_edge_length=float(NBHD_RADIUS)).create_simplex_tree(max_dimension=MAX_HOM_DIM+1); sq=False
    st.compute_persistence(homology_coeff_field=2,persistence_dim_max=True)
    o={}
    for d in range(MAX_HOM_DIM+1):
        b=np.asarray(st.persistence_intervals_in_dimension(d),dtype=np.float64).reshape(-1,2)
        o[d]=np.sqrt(np.maximum(b,0.0)) if (sq and len(b)) else b
    return o

def _vec(dg,pts):
    f=[]
    for d in range(MAX_HOM_DIM+1):
        bars=dg.get(d,np.zeros((0,2)))
        fin=bars[np.isfinite(bars).all(axis=1)] if len(bars) else bars.reshape(-1,2)
        if len(fin)==0: f.extend([0.0]*FPD); continue
        b,dt=fin[:,0],fin[:,1]; pers=np.maximum(dt-b,0.0); tot=float(pers.sum())
        ent=0.0
        if tot>0:
            p=pers/tot; nz=p[p>0]; ent=float(-(nz*np.log(nz)).sum())
        f.extend([float(len(fin)),tot,float(pers.max()),float(pers.mean()),float(pers.std()),
                  ent,float(b.mean()),float(dt.mean()),
                  *[float(((b<=r)&(dt>r)).sum()) for r in BETTI_RADII]])
    c=pts-pts.mean(0,keepdims=True)
    f.extend([float(len(pts)),float(np.sqrt((c**2).sum(1).mean())) if len(pts) else 0.0])
    return np.asarray(f,dtype=np.float32)

def topo_chain(ca,per_res):
    atoms=np.concatenate([a for a in per_res if len(a)],axis=0)
    tree=cKDTree(atoms); out=np.zeros((len(ca),TOPO_DIM),np.float32)
    for i,c in enumerate(ca):
        idx=tree.query_ball_point(c,NBHD_RADIUS)
        if len(idx)>MAX_NBHD_ATOMS: _,idx=tree.query(c,k=MAX_NBHD_ATOMS)
        pts=atoms[np.asarray(idx,dtype=int)]
        if len(pts)>=4: out[i]=_vec(_dg(pts),pts)[_TOPO_KEEP]
    return out

def dihedrals(pdb_id, chain_id, path, n_expect):
    try:
        with gzip.open(path,"rt") as fh:
            model=next(PDBParser(QUIET=True,PERMISSIVE=True).get_structure(pdb_id,fh).get_models())
        ch=model[chain_id]
    except Exception: return None
    residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
    if len(residues)!=n_expect: return None
    ppmap={}
    for pp in PPBuilder().build_peptides(ch, aa_only=False):
        for res,(phi,psi) in zip(pp, pp.get_phi_psi_list()): ppmap[res.id]=(phi,psi)
    phis,psis=[],[]
    for r in residues:
        phi,psi=ppmap.get(r.id,(None,None)); phis.append(phi); psis.append(psi)
    def enc(a):
        v=torch.tensor([x if x is not None else 0.0 for x in a],dtype=torch.float32)
        m=torch.tensor([x is not None for x in a],dtype=torch.float32)
        return torch.stack([torch.sin(v)*m,torch.cos(v)*m],1), m.unsqueeze(1)
    pf,pm=enc(phis); sf,sm=enc(psis)
    return torch.cat([pf,sf,pm,sm],1)

tok = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")
esm = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D").to(DEVICE).eval()

@torch.no_grad()
def embed_seq(seq):
    enc = tok([seq], return_tensors="pt", truncation=True, max_length=1002)
    enc = {k:v.to(DEVICE) for k,v in enc.items()}
    out = esm(**enc).last_hidden_state
    return out[0,1:1+len(seq),:].float().cpu()

t0=time.time(); built = {}   # key -> Data-like dict
n_chains_total = sum(len(c) for _,_,c in clean_units)
done=0
for src, pid, chains in clean_units:
    for cid, seq, ca, per_res, ligs in chains:
        n = len(ca)
        oh = torch.zeros(n, 21); oh[torch.arange(n), torch.tensor([A2I.get(c,20) for c in seq])]=1.0
        dih = dihedrals(pid, cid, paths[pid], n)
        if dih is None: continue
        esm_e = embed_seq(seq)
        topo = topo_chain(ca, per_res)
        pos = torch.from_numpy(ca); pos = pos - pos.mean(0, keepdim=True)
        x = torch.cat([oh, dih, esm_e, torch.from_numpy(topo.astype(np.float32))], dim=1)
        ei = rgc(pos)
        d_ = (pos[ei[0]]-pos[ei[1]]).norm(dim=1,keepdim=True)
        sep = (ei[0]-ei[1]).abs().float().unsqueeze(1)
        built[(src,pid,cid)] = dict(x=x, pos=pos, edge_index=ei,
                                    edge_attr=torch.cat([d_,torch.log1p(sep)],1),
                                    ca=ca, ligs=ligs)
        done += 1
    if done % 100 < len(chains): print(f"  {done}/{n_chains_total}  [{(time.time()-t0)/60:.1f}m]")

del esm; torch.cuda.empty_cache()
print(f"\nfeaturized {len(built)}/{n_chains_total} clean chains in {(time.time()-t0)/60:.1f} min")
torch.save({"built_keys": list(built.keys())}, WORK/"cache/built_index.pt")

## Cell 6 — Ensemble inference, cached per chain

Unlike Phase 12, per-residue probabilities (plus the coordinates and ground
truth needed to score them) are saved to disk here — so any *future* heuristic
idea can be tested in seconds instead of paying the ~3-hour featurization cost
again.

In [ ]:
mu, sd = blob6b["topo_mean"], blob6b["topo_std"]
N_SEQ_DIH_ESM = N_PRE

@torch.no_grad()
def predict_chain(entry):
    x = entry["x"].clone()
    x[:, N_SEQ_DIH_ESM:] = (x[:, N_SEQ_DIH_ESM:] - torch.from_numpy(mu)) / torch.from_numpy(sd)
    d = type("D", (), {})()
    d.x=x.to(DEVICE); d.pos=entry["pos"].to(DEVICE)
    d.edge_index=entry["edge_index"].to(DEVICE); d.edge_attr=entry["edge_attr"].to(DEVICE)
    probs = torch.stack([torch.sigmoid(m(d)) for m in ensemble], dim=0).mean(0)
    return probs.cpu().numpy()

t0=time.time()
cache = {}   # (src,pid,cid) -> {ca, edge_index(np), probs, ligs}
for key, entry in built.items():
    probs = predict_chain(entry)
    cache[key] = {"ca": entry["ca"], "edge_index": entry["edge_index"].numpy(),
                 "probs": probs, "ligs": entry["ligs"]}
print(f"inference on {len(cache)} chains in {(time.time()-t0)/60:.1f} min")

import pickle as _pkl
with open(WORK/"cache/phase13_per_chain_cache.pkl", "wb") as fh:
    _pkl.dump(cache, fh)
print(f"cached per-chain probabilities -> {WORK}/cache/phase13_per_chain_cache.pkl "
      f"({(WORK/'cache/phase13_per_chain_cache.pkl').stat().st_size/1e6:.0f} MB)")

## Cell 7 — Tune / test split, fixed before any heuristic is tried

Split at the **structure** level (not chain level, so a multi-chain HOLO4K
entry doesn't leak across the split), stratified by benchmark so both COACH420
and HOLO4K are represented in each half.

In [ ]:
structures = sorted({(src_, pid) for src_, pid, _cid in cache})
by_bench = defaultdict(list)
for s_, p_ in structures: by_bench[s_].append(p_)

rng = np.random.RandomState(2024)
tune_structs, test_structs = set(), set()
for s_, pids in by_bench.items():
    pids = sorted(pids); rng.shuffle(pids)
    n_tune = max(1, int(0.35 * len(pids)))
    tune_structs.update((s_, p) for p in pids[:n_tune])
    test_structs.update((s_, p) for p in pids[n_tune:])

print(f"tune structures: {len(tune_structs)}  |  test structures: {len(test_structs)}")
for s_ in by_bench:
    nt = sum(1 for x,p in tune_structs if x==s_); ns = sum(1 for x,p in test_structs if x==s_)
    print(f"  {s_:<10} tune {nt:>4}  test {ns:>4}")

def group_by_structure(struct_set):
    out = defaultdict(list)
    for (src_,pid,cid), entry in cache.items():
        if (src_,pid) in struct_set: out[(src_,pid)].append((cid, entry))
    return out

tune_groups = group_by_structure(tune_structs)
test_groups = group_by_structure(test_structs)

## Cell 8 — Candidate localization heuristics

In [ ]:
def dcc_dca(pred_center, ligs):
    if pred_center is None or not ligs: return None, None
    best_dcc, best_dca = np.inf, np.inf
    for lig_atoms in ligs:
        if len(lig_atoms) == 0: continue
        com = lig_atoms.mean(axis=0)
        best_dcc = min(best_dcc, float(np.linalg.norm(pred_center - com)))
        best_dca = min(best_dca, float(np.linalg.norm(lig_atoms - pred_center, axis=1).min()))
    return (best_dcc if np.isfinite(best_dcc) else None,
            best_dca if np.isfinite(best_dca) else None)


def heuristic_topk(ca, probs, k, weighted):
    if len(probs) == 0: return None, -np.inf
    k = min(k, len(probs))
    top = np.argsort(-probs)[:k]
    if weighted:
        w = probs[top]; w = w / max(w.sum(), 1e-9)
        centroid = (ca[top] * w[:, None]).sum(axis=0)
    else:
        centroid = ca[top].mean(axis=0)
    return centroid, float(probs[top].sum())


def heuristic_cc(ca, probs, edge_index, top_frac, min_prob, weighted):
    n = len(ca)
    thresh = max(min_prob, np.quantile(probs, 1 - top_frac))
    hot = np.where(probs >= thresh)[0]
    if len(hot) == 0: return None, -np.inf
    hot_set = set(hot.tolist())
    adj = defaultdict(list)
    for a, b in zip(edge_index[0], edge_index[1]):
        if a in hot_set and b in hot_set: adj[a].append(b); adj[b].append(a)
    seen = set(); best_centroid, best_score = None, -np.inf
    for start in hot:
        if start in seen: continue
        stack=[start]; comp=[]
        while stack:
            u=stack.pop()
            if u in seen: continue
            seen.add(u); comp.append(u)
            stack.extend(adj[u])
        comp = np.array(comp)
        score = float(probs[comp].sum())
        if score > best_score:
            best_score = score
            if weighted:
                w = probs[comp]; w = w / max(w.sum(), 1e-9)
                best_centroid = (ca[comp] * w[:, None]).sum(axis=0)
            else:
                best_centroid = ca[comp].mean(axis=0)
    return best_centroid, best_score


CANDIDATES = {
    "topk_1_unweighted":  lambda ca,p,ei: heuristic_topk(ca,p,1,False),
    "topk_3_weighted":    lambda ca,p,ei: heuristic_topk(ca,p,3,True),
    "topk_5_weighted":    lambda ca,p,ei: heuristic_topk(ca,p,5,True),
    "topk_10_weighted":   lambda ca,p,ei: heuristic_topk(ca,p,10,True),
    "topk_20_weighted":   lambda ca,p,ei: heuristic_topk(ca,p,20,True),
    "cc_phase12_baseline": lambda ca,p,ei: heuristic_cc(ca,p,ei,0.15,0.3,False),
    "cc_weighted_loose":   lambda ca,p,ei: heuristic_cc(ca,p,ei,0.15,0.3,True),
    "cc_weighted_tight":   lambda ca,p,ei: heuristic_cc(ca,p,ei,0.08,0.5,True),
}
print(f"{len(CANDIDATES)} candidate heuristics")

## Cell 9 — Score every candidate on TUNE only, pick the winner

In [ ]:
def score_group(groups, heuristic_fn):
    dcc_hits, dca_hits, n = 0, 0, 0
    for (src_, pid), chain_list in groups.items():
        all_cands = []
        ligs = chain_list[0][1]["ligs"]
        for cid, entry in chain_list:
            center, score = heuristic_fn(entry["ca"], entry["probs"], entry["edge_index"])
            if center is not None: all_cands.append((center, score))
        if not all_cands: continue
        all_cands.sort(key=lambda c: -c[1])
        dcc, dca = dcc_dca(all_cands[0][0], ligs)
        if dcc is None: continue
        n += 1
        dcc_hits += int(dcc < 4.0); dca_hits += int(dca < 4.0)
    return (dcc_hits/max(n,1), dca_hits/max(n,1), n)

print(f"{'heuristic':<24}{'tune DCC':>10}{'tune DCA':>10}{'n':>6}")
print("-"*50)
tune_scores = {}
for name, fn in CANDIDATES.items():
    dcc, dca, n = score_group(tune_groups, fn)
    tune_scores[name] = (dcc, dca, n)
    print(f"{name:<24}{dcc:>10.3f}{dca:>10.3f}{n:>6}")

best_name = max(tune_scores, key=lambda k: tune_scores[k][0])
print(f"\nselected on TUNE (best DCC): {best_name}  (DCC {tune_scores[best_name][0]:.3f})")

## Cell 10 — Final, one-shot evaluation on TEST

The winning heuristic and the Phase-12 baseline heuristic are both scored here,
**for the first time**, on the test structures neither ever influenced.

In [ ]:
baseline_fn = CANDIDATES["cc_phase12_baseline"]
winner_fn = CANDIDATES[best_name]

base_dcc, base_dca, base_n = score_group(test_groups, baseline_fn)
new_dcc, new_dca, new_n = score_group(test_groups, winner_fn)

# P2Rank reference: re-use Phase 12's already-computed numbers where the test
# structures overlap (P2Rank does not depend on our model, so re-running it is
# unnecessary -- report its full-clean-set number as the fixed comparison point).
P2RANK_REF = {
    "coach420": {"dcc": 0.4706, "dca": 0.6941},
    "holo4k":   {"dcc": 0.5509, "dca": 0.7796},
}

print(f"{'':<28}{'DCC':>8}{'DCA':>8}{'n':>6}")
print("-"*50)
print(f"{'Phase 12 baseline (test)':<28}{base_dcc:>8.3f}{base_dca:>8.3f}{base_n:>6}")
print(f"{'tuned: '+best_name:<28}{new_dcc:>8.3f}{new_dca:>8.3f}{new_n:>6}")
print(f"{'P2Rank (Phase 12, full clean set)':<28}{'--':>8}{'--':>8}{'':>6}  "
      f"(coach420 DCC {P2RANK_REF['coach420']['dcc']:.3f} / holo4k DCC {P2RANK_REF['holo4k']['dcc']:.3f})")

print(f"\nimprovement from tuning: DCC {new_dcc-base_dcc:+.3f}  DCA {new_dca-base_dca:+.3f}")
gap_before = base_dcc - np.mean([P2RANK_REF['coach420']['dcc'], P2RANK_REF['holo4k']['dcc']])
gap_after = new_dcc - np.mean([P2RANK_REF['coach420']['dcc'], P2RANK_REF['holo4k']['dcc']])
print(f"gap to P2Rank (rough, mixed benchmarks): before {gap_before:+.3f}  after {gap_after:+.3f}")

json.dump({"tune_scores": {k: list(v) for k,v in tune_scores.items()}, "selected": best_name,
           "test_baseline": {"dcc": base_dcc, "dca": base_dca, "n": base_n},
           "test_tuned": {"dcc": new_dcc, "dca": new_dca, "n": new_n},
           "p2rank_reference": P2RANK_REF},
          open(WORK/"phase13_summary.json", "w"), indent=2)
print("\nsaved -> phase13_summary.json")

---

## Reading this

**The comparison that matters is `Phase 12 baseline (test)` vs `tuned` — both
scored on the identical held-out structures, neither having influenced the
tuning choice.** That isolates exactly what changed: the conversion heuristic,
nothing else.

- **A clear DCC improvement** confirms the diagnosis: the per-residue signal was
  fine, the conversion step was the weak link, and this recovers real ground
  against P2Rank.
- **Little or no improvement** means the problem is deeper than the conversion
  heuristic — most likely that the per-residue probability map itself is not
  precise enough for tight single-point localization, which weighted-centroid
  tricks cannot fix. In that case the honest next step is training a dedicated
  pocket-center head end-to-end (how EquiPocket/VN-EGNN are actually built),
  not further heuristic tweaking.

Either way, `phase13_per_chain_cache.pkl` now holds every benchmark chain's
per-residue probabilities, so testing a ninth or tenth heuristic idea later
costs seconds, not another 3-hour run.